In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
import os
import time
import warnings
warnings.filterwarnings("ignore")

from ultralytics import YOLO

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
import ultralytics
print(f"Versao do Yolo: {ultralytics.__version__}")

yolo_custom = None


def detectar_yolo_custom(image_path, model, labels=[0,1], limiar_confianca=0.25):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inicio = time.time()
    resultados = model(
        img_rgb,
        conf=limiar_confianca,
        classes=labels,
        verbose=False
    )
    inferencia_ms = (time.time() - inicio) * 1000  # em ms
    retangulos = resultados[0].boxes
    deteccoes = []
    if retangulos is not None:
        for retang in retangulos:
            x1, y1, x2, y2 = retang.xyxy[0].cpu().numpy().astype(int)
            conf = float(retang.conf[0])
            deteccoes.append({"retang": (x1, y1, x2, y2), "conf": conf, 'cls': int(retang.cls.cpu().item())})
    
    return {
        "img": img_rgb,
        "deteccoes": deteccoes,
        "inferencia_ms": inferencia_ms,
        "num_deteccoes": len(deteccoes)
    }


def visualizar_deteccoes_yolo(result, title="YOLOv8 customizado"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    ax.imshow(result["img"])
    cores = plt.cm.Set1(np.linspace(0, 1, max(len(result["deteccoes"]), 1)))
    for i, det in enumerate(result["deteccoes"]):
        x1, y1, x2, y2 = det["retang"]
        conf = det["conf"]
        label_name = yolo_custom.names[det["cls"]]
        color = cores[i % len(cores)]
        # retangulo deteccao
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        # rotulo com confianca
        ax.text(
            x1, y1 - 6, f"{label_name.title()} {conf:.2f}",
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8)
        )
    
    ax.set_title(
        f"{title}\n"
        f"{result['num_deteccoes']} deteccoes | "
        f"Inferência: {result['inferencia_ms']:.1f}ms",
        fontsize=10, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    os.makedirs("output", exist_ok=True)
    plt.savefig("output/yolo_deteccoes.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Salvo em output/yolo_deteccoes.png")

In [ ]:
# carrega o melhor modelo treinado (v4 - yolov8n com freeze=10, 100 epocas completas)
yolo_custom = YOLO("runs/detect/transfer_v4_ep100_full/yolo_transfer_n/weights/best.pt")
print("Modelo carregado:", yolo_custom.info())

In [ ]:
# troca o nome da imagem para testar outras imagens do dataset/test/images/
image_path = "dataset/test/images/imagem_00101.jpg"

result = detectar_yolo_custom(image_path, yolo_custom)
visualizar_deteccoes_yolo(result)

In [ ]:
# testar em todas as imagens de teste e exibir resultados
test_images = sorted(Path('dataset/test/images').glob('*.jpg'))
print(f'Total de imagens de teste: {len(test_images)}')

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, img_path in enumerate(test_images):
    result = detectar_yolo_custom(str(img_path), yolo_custom)
    ax = axes[i]
    ax.imshow(result['img'])
    
    cores = plt.cm.Set1(np.linspace(0, 1, max(len(result['deteccoes']), 1)))
    for j, det in enumerate(result['deteccoes']):
        x1, y1, x2, y2 = det['retang']
        label_name = yolo_custom.names[det['cls']]
        color = cores[j % len(cores)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f"{label_name} {det['conf']:.2f}", color='white', fontsize=7,
                bbox=dict(facecolor=color, alpha=0.8, pad=1))
    
    ax.set_title(f"{img_path.name}\n{result['num_deteccoes']} deteccao(es)", fontsize=7)
    ax.axis('off')

plt.suptitle('Deteccoes YOLOv8n - Conjunto de Teste', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/todas_deteccoes_teste.png', dpi=100, bbox_inches='tight')
plt.show()
print('Salvo em output/todas_deteccoes_teste.png')